# RISK ANALYTICS — 03 Capa Silver

**Origen:** `eventosriesgoyvar.bronze.eventos_riesgo_raw` (51.889 filas × 61 columnas: 58 de origen + 3 de trazabilidad)
**Destino:** `eventosriesgoyvar.silver.eventos_riesgo` (Delta)

Reglas de esta capa:
- **Sin deduplicación y sin eliminación:** las 51.889 filas se conservan íntegras. Las filas con valores idénticos son eventos distintos (regla acordada en la inspección de la fuente); no se marcan como duplicados ni se eliminan.
- **Sin imputación:** los ausentes de la fuente ('-', cadena vacía, nulo) pasan a NULL. Nunca se inventan valores para completar campos.
- **Estandarización:** nombres de columna semánticos (mapeo documentado al final), fechas `date`, horas/minutos `int`, montos `decimal(18,2)`, categorías homologadas a los valores reales de la base.
- **Montos:** se tipan sin alterar su valor; negativos, ceros y extremos se revisan y marcan con banderas, no se corrigen.
- **Trazabilidad:** `id_evento` único por fila, banderas de calidad `flg_*`, y las columnas de origen Bronze (`_archivo_origen`, `_fecha_ingesta`, `_lote_procesamiento`) se conservan junto con `_lote_silver`.
- **Reconstrucción idempotente:** la tabla Silver se sobrescribe completa en cada ejecución; Bronze queda intacta como respaldo de los valores crudos.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
from itertools import chain

TABLA_BRONZE = "eventosriesgoyvar.bronze.eventos_riesgo_raw"
TABLA_SILVER = "eventosriesgoyvar.silver.eventos_riesgo"
LOTE_SILVER = f"silver_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

spark.sql("CREATE SCHEMA IF NOT EXISTS eventosriesgoyvar.silver "
          "COMMENT 'Capa Silver: eventos de riesgo operacional estandarizados y con banderas de calidad'")

bronze = spark.table(TABLA_BRONZE)
FILAS_BRONZE = bronze.count()
print("Filas en Bronze:", FILAS_BRONZE)
print("Tabla destino:", TABLA_SILVER)
print("Lote Silver:", LOTE_SILVER)

Filas en Bronze: 51889
Tabla destino: eventosriesgoyvar.silver.eventos_riesgo
Lote Silver: silver_20261121_210607


In [0]:
# Funciones de estandarización aplicadas a los VALORES crudos de Bronze.
# Regla general: los ausentes de la fuente ('-', cadena vacía, nulo) pasan a NULL; nunca se imputan.

def texto(nombre):
    """Texto homologado: se retira el espacio de no-ruptura (\xa0) detectado en la fuente
    (2.243 filas de codproceso traían '\\xa010.2'), trim; '-', vacío y nulo -> NULL."""
    t = F.trim(F.regexp_replace(F.col(nombre), "\u00a0", " "))
    return F.when(t.isNull() | (t == "-") | (t == ""), F.lit(None).cast("string")).otherwise(t)

def monto(nombre):
    """Monto tipado: '-', vacío y nulo -> NULL; se retira el '.' separador de miles
    y se tipa como decimal(18,2) (los valores reales de la base son enteros en COP)."""
    t = F.trim(F.col(nombre))
    return (F.when(t.isNull() | (t == "-") | (t == ""), F.lit(None).cast("decimal(18,2)"))
             .otherwise(F.regexp_replace(t, r"\.", "").cast("decimal(18,2)")))

def fecha(nombre):
    """Fecha tipada: acepta dd/MM/yyyy y d/M/yyyy (sin cero inicial, p. ej. 2/03/2026).
    Excepción real de la fuente: 109 fechas de RECUPERACIÓN vienen en MM/dd/yyyy
    (p. ej. 02/20/2026). Se interpretan como MM/dd SOLO cuando el componente central
    no puede ser mes (>12); el resto queda en dd/MM, formato dominante de la base.
    '-', vacío y nulo -> NULL."""
    t = F.trim(F.col(nombre))
    es_mmdd = F.split(t, "/")[1].cast("int") > 12
    return (F.when(t.isNull() | (t == "-") | (t == ""), F.lit(None).cast("date"))
             .when(es_mmdd, F.to_date(t, "M/d/yyyy"))
             .otherwise(F.to_date(t, "d/M/yyyy")))

def entero(nombre):
    """Hora/minuto tipado como int; '-', vacío y nulo -> NULL."""
    t = F.trim(F.col(nombre))
    return (F.when(t.isNull() | (t == "-") | (t == ""), F.lit(None).cast("int"))
             .otherwise(t.cast("int")))

print("Funciones de estandarización definidas.")

Funciones de estandarización definidas.


In [0]:
# --- Homologación de dimensiones sobre los valores REALES de la base ---

# 1) CANAL: las 13 grafías reales -> 7 canales canónicos.
#    Unificación de grafía (mayúsculas/tildes) más dos sinónimos reales de la base:
#    'APLICACIÓN MOVIL' -> 'APLICACIÓN MÓVIL' y 'CAJERO AUTOMÁTICO' -> 'CAJERO ELECTRÓNICO' (1 fila).
CLAVE_CANAL = {
    "OTROS": "OTROS", "AGENCIA": "AGENCIA", "CAJERO ELECTRONICO": "CAJERO ELECTRÓNICO",
    "CAJERO AUTOMATICO": "CAJERO ELECTRÓNICO",  # sinónimo real de la base (1 fila)
    "INTERNET": "INTERNET", "POS": "POS", "APLICACION MOVIL": "APLICACIÓN MÓVIL", "IVR": "IVR",
}
MAPA_CANAL = F.create_map([F.lit(x) for x in chain(*CLAVE_CANAL.items())])

def clave_canal_expr():
    """Clave de homologación: mayúsculas, sin tildes, espacios simples (incl. \xa0)."""
    s = F.upper(F.translate(F.trim(F.regexp_replace(F.col("canal"), "\u00a0", " ")), "ÁÉÍÓÚ", "AEIOU"))
    return F.regexp_replace(s, r"\s+", " ")

# 2) TIPO / MONEDA / CÓDIGO PÉRDIDA / CLASE DE RIESGO: trim + mayúsculas.
#    Corrige los casos reales detectados: tipo 'p' (2 filas) y tipo/código con espacios
#    ('R ', 'A '), moneda 'Pesos' (19 filas). El filtro de VaR se aplica SIEMPRE sobre
#    estos valores normalizados (regla acordada: VaR solo con CODIGOPERDIDA = 'A').

# 3) PROCESO y ACTIVIDAD: solo se unifican variantes de ESCRITURA de un mismo nombre
#    (mayúsculas/minúsculas, tildes, punto final, espacios múltiples); la grafía canónica
#    es la más frecuente de cada grupo. Nombres genuinamente distintos NO se unifican.
def clave_norm_expr(colname):
    s = F.upper(F.translate(F.trim(F.regexp_replace(F.col(colname), "\u00a0", " ")), "ÁÉÍÓÚ", "AEIOU"))
    s = F.regexp_replace(s, r"\s+", " ")
    s = F.regexp_replace(s, r"\.+$", "")
    return F.trim(s)

def variantes_canonicas(df, colname):
    """Clave normalizada -> grafía más frecuente (unifica solo variantes de escritura)."""
    g = (df.select(clave_norm_expr(colname).alias("clave"), F.trim(F.col(colname)).alias("valor"))
           .filter(F.col("valor").isNotNull() & (F.trim(F.col(colname)) != "-"))
           .groupBy("clave", "valor").count())
    mejor = {}
    for r in g.collect():
        k, v, c = r["clave"], r["valor"], r["count"]
        if k not in mejor or c > mejor[k][1]:
            mejor[k] = (v, c)
    return {k: v for k, (v, c) in mejor.items()}

MAPA_PROCESO = variantes_canonicas(bronze, "proceso")
MAPA_ACTIVIDAD = variantes_canonicas(bronze, "actividad")
print(f"Procesos: {len(MAPA_PROCESO)} grafías canónicas | Actividades: {len(MAPA_ACTIVIDAD)} grafías canónicas")

mapa_proceso = F.create_map([F.lit(x) for x in chain(*MAPA_PROCESO.items())])
mapa_actividad = F.create_map([F.lit(x) for x in chain(*MAPA_ACTIVIDAD.items())])

# Códigos que siguen asociados a más de un nombre tras la homologación: NO se reetiquetan
# (no se inventa la correspondencia correcta), se marcan con bandera de ambigüedad.
def codigos_ambiguos(df, colcod, colnom, mapa):
    m = F.element_at(mapa, clave_norm_expr(colnom))
    # normaliza el espacio de no-ruptura: '\xa010.2' -> '10.2'
    cod = F.trim(F.regexp_replace(F.col(colcod), "\u00a0", " "))
    d = df.select(cod.alias("cod"), m.alias("nom")).distinct()
    return [r["cod"] for r in d.groupBy("cod").count().filter(F.col("count") > 1).select("cod").collect()]

CODS_PROCESO_AMBIGUO = codigos_ambiguos(bronze, "codproceso", "proceso", mapa_proceso)
CODS_ACTIVIDAD_AMBIGUA = codigos_ambiguos(bronze, "codactiviadad", "actividad", mapa_actividad)
print("codproceso con más de un proceso:", CODS_PROCESO_AMBIGUO)
print("codactiviadad con más de una actividad:", CODS_ACTIVIDAD_AMBIGUA)

Procesos: 36 grafías canónicas | Actividades: 117 grafías canónicas
codproceso con más de un proceso: ['10.2', '6.3', '11.3']
codactiviadad con más de una actividad: ['6.4.11', '6.4.17', '6.4.6', '11.1.2', '1.2.2', '11.1.4']


In [0]:
# --- Cortes de severidad calculados sobre los valores reales de la cuantía bruta ---
# La base NO trae columna de severidad: el nivel de severidad es una variable DERIVADA
# de la cuantía bruta (pérdida registrada), con cortes por percentiles de la propia base.
# Percentiles EXACTOS (F.percentile): con 26.701 valores el cálculo exacto es inmediato
# y elimina la desviación de percentile_approx (p50 aprox = 367.000 vs exacto = 367.371).
pc = (bronze.select(monto("cuantia_bruta").alias("b")).filter(F.col("b").isNotNull())
        .agg(F.percentile("b", 0.50).alias("p50"),
             F.percentile("b", 0.90).alias("p90"),
             F.percentile("b", 0.99).alias("p99")).collect()[0])
P50, P90, P99 = float(pc["p50"]), float(pc["p90"]), float(pc["p99"])
print(f"Cortes de severidad (COP, sobre 26.701 cuantías brutas): "
      f"BAJA < {P50:,.0f} | MEDIA >= {P50:,.0f} | ALTA >= {P90:,.0f} | EXTREMA >= {P99:,.0f}")

# --- Construcción de la base Silver: nombres, tipos y homologación ---
df_silver = bronze.select(
    # Identificación del registro (no existe clave natural única en la base)
    texto("idinsumo").alias("idinsumo"),

    # Tiempo del evento (fechas date, horas/minutos int; ausentes -> NULL)
    fecha("fechainicio").alias("fecha_inicio"),
    entero("horainicio").alias("hora_inicio"),
    entero("minutoinicio").alias("minuto_inicio"),
    fecha("fechafinal").alias("fecha_fin"),
    entero("horafin").alias("hora_fin"),
    entero("minutofin").alias("minuto_fin"),
    fecha("fechadescubrimiento").alias("fecha_descubrimiento"),
    entero("horades").alias("hora_descubrimiento"),
    entero("minutodes").alias("minuto_descubrimiento"),
    fecha("fecha_contabilizacion").alias("fecha_contabilizacion"),
    entero("horacon").alias("hora_contabilizacion"),
    entero("minutocon").alias("minuto_contabilizacion"),
    fecha("fecha_recuperaci_n").alias("fecha_recuperacion"),
    entero("horarec").alias("hora_recuperacion"),
    entero("minutorec").alias("minuto_recuperacion"),

    # Clasificación monetaria del evento
    F.upper(F.trim(F.col("tipo"))).alias("tipo_evento"),          # 'p' -> 'P'
    F.trim(F.col("codigoperdida")).alias("codigo_perdida"),      # 'A ' -> 'A'
    F.upper(F.trim(F.col("moneda"))).alias("moneda"),             # 'Pesos' -> 'PESOS'
    texto("clase_de_riesgo").alias("clase_de_riesgo"),

    # Cuantías (decimal(18,2); '-' y vacío -> NULL, valor intacto)
    monto("cuantia_bruta").alias("cuantia_bruta"),
    monto("cuantia_total_recuperada").alias("cuantia_total_recuperada"),
    monto("cuantia_recuperada_seguros").alias("cuantia_recuperada_seguros"),
    monto("cuantia_otras_recuperaciones").alias("cuantia_otras_recuperaciones"),
    monto("cuantia_neta").alias("cuantia_neta"),

    # Taxonomía del riesgo operacional (niveles N1/N2/N3)
    texto("cod_categoria_n1").alias("cod_categoria_n1"),
    texto("categoria_n1").alias("categoria_n1"),
    texto("codigo_subcategoria_n2").alias("codigo_subcategoria_n2"),
    texto("subcategoria_n2").alias("subcategoria_n2"),
    texto("codigo_subcategoria_n3").alias("codigo_subcategoria_n3"),
    texto("subcategoria_riesgo_n3").alias("subcategoria_riesgo_n3"),

    # Producto, geografía y canal
    texto("codigoproducto").alias("codigo_producto"),
    texto("producto").alias("producto"),
    texto("codigosisgeo").alias("codigo_sisgeo"),
    texto("ciudad").alias("ciudad"),
    texto("codigosucursal").alias("codigo_sucursal"),
    texto("sucursal").alias("sucursal"),
    F.coalesce(F.element_at(MAPA_CANAL, clave_canal_expr()), texto("canal")).alias("canal"),

    # Impacto y fuente del evento
    texto("codigoimpacto").alias("codigo_impacto"),
    texto("impacto").alias("impacto"),
    texto("codigofuente").alias("codigo_fuente"),
    texto("fuente39").alias("fuente_evento"),                    # FUENTE del evento (col. 39)

    # Líneas operativas
    texto("codigo_linea_operativa_n1").alias("codigo_linea_operativa_n1"),
    texto("linea_operativa_n1").alias("linea_operativa_n1"),
    texto("codigo_linea_operativa_n2").alias("codigo_linea_operativa_n2"),
    texto("linea_operativa_n2").alias("linea_operativa_n2"),

    # Estructura, proceso y actividad
    texto("codestror").alias("codigo_estructura_ror"),
    texto("codproceso").alias("codigo_proceso"),
    F.coalesce(F.element_at(mapa_proceso, clave_norm_expr("proceso")), texto("proceso")).alias("proceso"),
    texto("codactiviadad").alias("codigo_actividad"),           # typo corregido en el NOMBRE
    F.coalesce(F.element_at(mapa_actividad, clave_norm_expr("actividad")), texto("actividad")).alias("actividad"),

    # Causas (solo códigos: la base no trae nombre descriptivo de causa)
    texto("codriesgo").alias("cod_riesgo"),
    texto("codcausa").alias("cod_causa"),

    # Cuentas y documentos
    texto("cuenta_p_rdida_8_digitos").alias("cuenta_perdida"),
    texto("documento_p_rdida").alias("documento_perdida"),
    texto("cuenta_recuperaci_n_8_digitos").alias("cuenta_recuperacion"),
    texto("documento_recuperaci_n").alias("documento_recuperacion"),

    # Origen del registro (col. 56: distinta de la FUENTE del evento)
    texto("fuente56").alias("origen_registro"),

    # Trazabilidad Bronze
    "_archivo_origen", "_fecha_ingesta", "_lote_procesamiento",
)
print("Columnas base Silver:", len(df_silver.columns))

Cortes de severidad (COP, sobre 26.701 cuantías brutas): BAJA < 367,371 | MEDIA >= 367,371 | ALTA >= 14,217,315 | EXTREMA >= 152,000,000
Columnas base Silver: 61


In [0]:
# --- Banderas de calidad y trazabilidad: los VALORES nunca se modifican ---
FLG = [
    ("flg_sin_cuantia_bruta", F.col("cuantia_bruta").isNull()),
    ("flg_sin_cuantia_neta", F.col("cuantia_neta").isNull()),
    ("flg_bruta_sin_neta", F.col("cuantia_bruta").isNotNull() & F.col("cuantia_neta").isNull()),
    ("flg_neta_sin_bruta", F.col("cuantia_neta").isNotNull() & F.col("cuantia_bruta").isNull()),
    ("flg_neta_negativa", F.col("cuantia_neta") < 0),
    ("flg_con_recuperacion", F.col("cuantia_total_recuperada").isNotNull()
        | F.col("cuantia_recuperada_seguros").isNotNull()
        | F.col("cuantia_otras_recuperaciones").isNotNull()),
    ("flg_monto_extremo", F.col("cuantia_bruta") >= P99),
    ("flg_inconsistencia_neta", F.col("cuantia_bruta").isNotNull() & F.col("cuantia_neta").isNotNull()
        & (F.col("cuantia_neta") != F.col("cuantia_bruta")
           - F.coalesce(F.col("cuantia_total_recuperada").cast("decimal(18,2)"), F.lit(0).cast("decimal(18,2)")))),
    ("flg_proceso_ambiguo", F.col("codigo_proceso").isin(CODS_PROCESO_AMBIGUO) if CODS_PROCESO_AMBIGUO else F.lit(False)),
    ("flg_actividad_ambigua", F.col("codigo_actividad").isin(CODS_ACTIVIDAD_AMBIGUA) if CODS_ACTIVIDAD_AMBIGUA else F.lit(False)),
    ("flg_sin_idinsumo", F.col("idinsumo").isNull()),
]
for nombre, expr in FLG:
    df_silver = df_silver.withColumn(nombre, expr)

# Requiere revisión: excepciones operativas detectadas. La ausencia estructural de cuantía
# (48,5% de los eventos) es una limitación de la fuente, no una anomalía del registro.
df_silver = df_silver.withColumn(
    "flg_requiere_revision",
    F.col("flg_neta_negativa") | F.col("flg_monto_extremo") | F.col("flg_inconsistencia_neta")
    | F.col("flg_neta_sin_bruta") | F.col("flg_proceso_ambiguo") | F.col("flg_actividad_ambigua"))

# Nivel de severidad DERIVADO de la cuantía bruta (cortes p50/p90/p99 de la propia base);
# NULL cuando no hay cuantía (ausencia de cuantía NO se imputa como severidad BAJA ni como 0).
df_silver = df_silver.withColumn(
    "nivel_severidad",
    F.when(F.col("cuantia_bruta").isNull(), F.lit(None).cast("string"))
     .when(F.col("cuantia_bruta") >= P99, "EXTREMA")
     .when(F.col("cuantia_bruta") >= P90, "ALTA")
     .when(F.col("cuantia_bruta") >= P50, "MEDIA")
     .otherwise("BAJA"))

# Campos derivados de la fecha del evento
df_silver = (df_silver
    .withColumn("anio_evento", F.year("fecha_inicio").cast("int"))
    .withColumn("mes_evento", F.month("fecha_inicio").cast("int")))

# id_evento: identificador único por fila (la base no tiene clave natural única).
# Orden determinista por campos de negocio; entre filas idénticas el orden lo arbitra Spark,
# pero cada evento conserva su propio id y ninguna fila se elimina ni se fusiona.
w_id = Window.orderBy("fecha_inicio", "hora_inicio", "minuto_inicio", "ciudad", "producto",
                      "codigo_proceso", "cuantia_bruta", "idinsumo")
df_silver = (df_silver
    .withColumn("id_evento", F.row_number().over(w_id).cast("long"))
    .withColumn("_lote_silver", F.lit(LOTE_SILVER)))

print("Columnas finales Silver:", len(df_silver.columns))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Columnas finales Silver: 78


In [0]:
# --- Orden final de columnas y escritura como tabla Delta ---
orden = (
    ["id_evento", "idinsumo"]
    + ["fecha_inicio", "hora_inicio", "minuto_inicio", "fecha_fin", "hora_fin", "minuto_fin",
       "fecha_descubrimiento", "hora_descubrimiento", "minuto_descubrimiento",
       "fecha_contabilizacion", "hora_contabilizacion", "minuto_contabilizacion",
       "fecha_recuperacion", "hora_recuperacion", "minuto_recuperacion",
       "anio_evento", "mes_evento"]
    + ["tipo_evento", "codigo_perdida", "moneda", "clase_de_riesgo"]
    + ["cuantia_bruta", "cuantia_total_recuperada", "cuantia_recuperada_seguros",
       "cuantia_otras_recuperaciones", "cuantia_neta", "nivel_severidad"]
    + ["cod_categoria_n1", "categoria_n1", "codigo_subcategoria_n2", "subcategoria_n2",
       "codigo_subcategoria_n3", "subcategoria_riesgo_n3"]
    + ["codigo_producto", "producto", "codigo_sisgeo", "ciudad", "codigo_sucursal", "sucursal", "canal"]
    + ["codigo_impacto", "impacto", "codigo_fuente", "fuente_evento",
       "codigo_linea_operativa_n1", "linea_operativa_n1", "codigo_linea_operativa_n2", "linea_operativa_n2"]
    + ["codigo_estructura_ror", "codigo_proceso", "proceso", "codigo_actividad", "actividad",
       "cod_riesgo", "cod_causa"]
    + ["cuenta_perdida", "documento_perdida", "cuenta_recuperacion", "documento_recuperacion",
       "origen_registro"]
    + [n for n, _ in FLG] + ["flg_requiere_revision"]
    + ["_archivo_origen", "_fecha_ingesta", "_lote_procesamiento", "_lote_silver"]
)
df_final = df_silver.select(orden)
print("Columnas a escribir:", len(orden))

df_final.write.mode("overwrite").saveAsTable(TABLA_SILVER)
spark.sql(f"COMMENT ON TABLE {TABLA_SILVER} IS "
          "'Capa Silver: eventos de riesgo operacional estandarizados, sin deduplicacion, "
          "con banderas de calidad y severidad derivada de la cuantia bruta (cortes p50/p90/p99)'")

silver = spark.table(TABLA_SILVER)
print("Tabla escrita:", TABLA_SILVER, "|", silver.count(), "filas |", len(silver.columns), "columnas")

Columnas a escribir: 78
Tabla escrita: eventosriesgoyvar.silver.eventos_riesgo | 51889 filas | 78 columnas


In [0]:
# --- Validaciones: Silver contra Bronze y contra los conteos verificados en la inspección ---
todo_ok = True

def check(nombre, ok, detalle=""):
    global todo_ok
    todo_ok = todo_ok and ok
    print(("OK   " if ok else "FALLA ") + nombre + (" | " + detalle if detalle else ""))

# 1) Conservación de registros (sin deduplicación, sin eliminación)
FILAS_SILVER = silver.count()
check("1. Filas Bronze == Silver == 51.889", FILAS_SILVER == FILAS_BRONZE == 51889,
      f"bronze={FILAS_BRONZE}, silver={FILAS_SILVER}")

# 2) Fechas: todas las presentes en la fuente quedaron parseadas (ninguna fecha válida perdida)
for c in ["fecha_inicio", "fecha_fin", "fecha_descubrimiento"]:
    nulos = silver.filter(F.col(c).isNull()).count()
    check(f"2. {c}: 100% parseada", nulos == 0, f"nulas={nulos}")
for c, esperadas in [("fecha_contabilizacion", 2057), ("fecha_recuperacion", 235)]:
    presentes = silver.filter(F.col(c).isNotNull()).count()
    check(f"2. {c}: {esperadas} presentes, todas parseadas (recuperación incluye 109 en MM/dd)",
          presentes == esperadas, f"presentes={presentes}")
print()

# 3) Dominios homologados
vals_tipo = sorted([r["tipo_evento"] for r in silver.select("tipo_evento").distinct().collect()])
check("3a. tipo_evento ∈ {P, R}", vals_tipo == ["P", "R"], str(vals_tipo))
check("3b. moneda: solo PESOS", silver.select("moneda").distinct().count() == 1)
canal_counts = silver.groupBy("canal").count().orderBy(F.desc("count")).collect()
check("3c. canal: 7 valores canónicos, sin grafías residuales", len(canal_counts) == 7,
      str([(r["canal"], r["count"]) for r in canal_counts]))
mal_alin = silver.filter(~(((F.col("tipo_evento") == "P") & (F.col("codigo_perdida") == "B"))
                          | ((F.col("tipo_evento") == "R") & (F.col("codigo_perdida") == "A")))).count()
check("3d. tipo_evento <-> codigo_perdida alineados (P-B, R-A)", mal_alin == 0, f"violaciones={mal_alin}")

# 4) Montos: negativos, ceros y extremos (valores intactos, solo marcados)
cuantias = ["cuantia_bruta", "cuantia_total_recuperada", "cuantia_recuperada_seguros",
            "cuantia_otras_recuperaciones", "cuantia_neta"]
neg = silver.agg(*[F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(c) for c in cuantias]).collect()[0].asDict()
check("4a. Negativos solo en cuantia_neta (28, se conservan y se marcan)",
      neg["cuantia_neta"] == 28 and all(neg[c] == 0 for c in cuantias if c != "cuantia_neta"), str(neg))
ceros = silver.filter((F.col("cuantia_bruta") == 0) | (F.col("cuantia_neta") == 0)).count()
check("4b. Sin montos en cero", ceros == 0, f"ceros={ceros}")

# 5) Disponibilidad de cuantías (conteos verificados en la inspección de la fuente)
disp = silver.agg(
    F.count(F.when(F.col("cuantia_bruta").isNotNull(), 1)).alias("bruta"),
    F.count(F.when(F.col("cuantia_neta").isNotNull(), 1)).alias("neta"),
    F.count(F.when(F.col("cuantia_bruta").isNotNull() & F.col("cuantia_neta").isNotNull(), 1)).alias("ambas"),
    F.count(F.when(F.col("cuantia_bruta").isNotNull() & F.col("cuantia_neta").isNull(), 1)).alias("bruta_sin_neta"),
    F.count(F.when(F.col("cuantia_neta").isNotNull() & F.col("cuantia_bruta").isNull(), 1)).alias("neta_sin_bruta"),
    F.count(F.when(F.col("flg_con_recuperacion"), 1)).alias("recuperaciones")).collect()[0].asDict()
check("5. Disponibilidad de cuantías", (disp["bruta"], disp["neta"], disp["ambas"], disp["bruta_sin_neta"],
      disp["neta_sin_bruta"]) == (26701, 26588, 26567, 134, 21), str(disp))

# 6) Banderas de calidad
flg_counts = silver.agg(*[F.sum(F.when(F.col(n), 1).otherwise(0)).alias(n) for n, _ in FLG]) \
                  .collect()[0].asDict()
for n, _ in FLG:
    print(f"  {n}: {flg_counts[n]}")
check("6a. flg_neta_negativa == 28", flg_counts["flg_neta_negativa"] == 28)
check("6b. flg_sin_cuantia_bruta == 25.188", flg_counts["flg_sin_cuantia_bruta"] == 25188)
check("6c. flg_sin_idinsumo == 23.026", flg_counts["flg_sin_idinsumo"] == 23026)

# 7) Severidad derivada
sev = {r["nivel_severidad"]: r["count"] for r in silver.groupBy("nivel_severidad").count().collect()}
check("7. Severidad cubre exactamente los 26.701 eventos con cuantía bruta",
      sev.get("BAJA", 0) + sev.get("MEDIA", 0) + sev.get("ALTA", 0) + sev.get("EXTREMA", 0) == 26701
      and silver.filter(F.col("nivel_severidad").isNull()).count() == 25188, str(sev))

# 8) id_evento único y continuo
ids = silver.agg(F.countDistinct("id_evento").alias("d"), F.min("id_evento").alias("mn"),
                  F.max("id_evento").alias("mx")).collect()[0].asDict()
check("8. id_evento único, 1..51.889", ids["d"] == 51889 and ids["mn"] == 1 and ids["mx"] == 51889, str(ids))

# 9) Horas y minutos en rango (verificado en la fuente: 0-23 y 0-59)
horas = [c for c in silver.columns if c.startswith("hora_")]
minutos = [c for c in silver.columns if c.startswith("minuto_")]
horas_fuera = silver.filter(F.greatest(*[F.when(F.col(c) > 23, 1).otherwise(0) for c in horas]) > 0).count()
minutos_fuera = silver.filter(F.greatest(*[F.when((F.col(c) < 0) | (F.col(c) > 59), 1).otherwise(0) for c in minutos]) > 0).count()
check("9. Horas 0-23 y minutos 0-59", horas_fuera == 0 and minutos_fuera == 0,
      f"horas>23={horas_fuera}, minutos fuera de 0-59={minutos_fuera}")

# 10) Sin espacios residuales en dimensiones de texto
cols_dim = ["categoria_n1", "subcategoria_n2", "subcategoria_riesgo_n3", "producto", "ciudad",
            "sucursal", "canal", "impacto", "fuente_evento", "linea_operativa_n1",
            "linea_operativa_n2", "proceso", "actividad", "clase_de_riesgo", "origen_registro"]
resid = silver.filter(F.greatest(*[F.when(F.length(F.trim(F.col(c))) != F.length(F.col(c)), 1).otherwise(0)
                                    for c in cols_dim]) > 0).count()
check("10. Textos sin espacios residuales", resid == 0, f"con residuos={resid}")
nbsp = silver.filter(F.greatest(*[F.when(F.col(c).contains("\u00a0"), 1).otherwise(0)
                                   for c in ["codigo_proceso", "proceso", "canal", "actividad", "codigo_actividad"]]) > 0).count()
check("10b. Sin espacios de no-ruptura (\\xa0) residuales", nbsp == 0, f"con \\xa0={nbsp}")

print()
print("RESULTADO GLOBAL:", "TODAS LAS VALIDACIONES OK" if todo_ok else "HAY VALIDACIONES FALLADAS")

OK   1. Filas Bronze == Silver == 51.889 | bronze=51889, silver=51889
OK   2. fecha_inicio: 100% parseada | nulas=0
OK   2. fecha_fin: 100% parseada | nulas=0
OK   2. fecha_descubrimiento: 100% parseada | nulas=0
OK   2. fecha_contabilizacion: 2057 presentes, todas parseadas (recuperación incluye 109 en MM/dd) | presentes=2057
OK   2. fecha_recuperacion: 235 presentes, todas parseadas (recuperación incluye 109 en MM/dd) | presentes=235

OK   3a. tipo_evento ∈ {P, R} | ['P', 'R']
OK   3b. moneda: solo PESOS
OK   3c. canal: 7 valores canónicos, sin grafías residuales | [('OTROS', 28161), ('AGENCIA', 11962), ('CAJERO ELECTRÓNICO', 6867), ('INTERNET', 3824), ('POS', 779), ('APLICACIÓN MÓVIL', 240), ('IVR', 56)]
OK   3d. tipo_evento <-> codigo_perdida alineados (P-B, R-A) | violaciones=0
OK   4a. Negativos solo en cuantia_neta (28, se conservan y se marcan) | {'cuantia_bruta': 0, 'cuantia_total_recuperada': 0, 'cuantia_recuperada_seguros': 0, 'cuantia_otras_recuperaciones': 0, 'cuantia_neta

In [0]:
# --- Resumen de calidad para el reporte de la capa ---
resumen_general = silver.agg(
    F.count("*").alias("eventos"),
    F.count(F.when(F.col("cuantia_bruta").isNotNull(), 1)).alias("con_cuantia_bruta"),
    F.count(F.when(F.col("cuantia_neta").isNotNull(), 1)).alias("con_cuantia_neta"),
    F.count(F.when(F.col("flg_con_recuperacion"), 1)).alias("con_recuperacion"),
    F.count(F.when(F.col("flg_requiere_revision"), 1)).alias("requieren_revision"),
    F.round(F.sum("cuantia_bruta"), 0).alias("suma_cuantia_bruta_cop"),
    F.min("fecha_inicio").alias("primera_fecha"),
    F.max("fecha_inicio").alias("ultima_fecha")).collect()[0].asDict()
display(spark.createDataFrame([resumen_general]))

print("Distribución de severidad (derivada de cuantía bruta):")
display(silver.groupBy("nivel_severidad")
        .agg(F.count("*").alias("eventos"), F.round(F.sum("cuantia_bruta"), 0).alias("suma_bruta_cop"))
        .orderBy(F.desc("eventos")))

print("Canal homologado:")
display(silver.groupBy("canal").count().orderBy(F.desc("count")))

con_cuantia_bruta,con_cuantia_neta,con_recuperacion,eventos,primera_fecha,requieren_revision,suma_cuantia_bruta_cop,ultima_fecha
26701,26588,235,51889,2014-12-05,15888,241789756996.000000000000000000,2026-09-30


Distribución de severidad (derivada de cuantía bruta):


nivel_severidad,eventos,suma_bruta_cop
null,25188,null
BAJA,13350,1499873516
MEDIA,10680,32828084627
ALTA,2403,92140893834
EXTREMA,268,115320905019


Canal homologado:


canal,count
OTROS,28161
AGENCIA,11962
CAJERO ELECTRÓNICO,6867
INTERNET,3824
POS,779
APLICACIÓN MÓVIL,240
IVR,56


## Registro de decisiones de la capa Silver (20-Nov-2026)

### 1. Nombres de columnas (mapeo completo)
| Bronze (snake_case crudo) | Silver | Nota |
| --- | --- | --- |
| fechainicio/horainicio/minutoinicio | fecha_inicio / hora_inicio / minuto_inicio | date / int / int |
| fechafinal/horafin/minutofin | fecha_fin / hora_fin / minuto_fin | |
| fechadescubrimiento/horades/minutodes | fecha_descubrimiento / hora_descubrimiento / minuto_descubrimiento | |
| fecha_contabilizacion/horacon/minutocon | fecha_contabilizacion / hora_contabilizacion / minuto_contabilizacion | |
| fecha_recuperaci_n/horarec/minutorec | fecha_recuperacion / hora_recuperacion / minuto_recuperacion | |
| tipo | tipo_evento | 'p'→'P', espacios fuera (108 filas 'R ') |
| codigoperdida | codigo_perdida | trim ('A '→'A', 2 filas) |
| moneda | moneda | 'Pesos'→'PESOS' (19 filas) |
| cuantia_* (5 columnas) | mismo nombre | decimal(18,2); '-'→NULL; '.' de miles retirado |
| codigoproducto / codigosisgeo / codigosucursal / codigosucursal | codigo_producto / codigo_sisgeo / codigo_sucursal | prefijo legible |
| canal | canal | 13 grafías → 7 canónicos |
| codigoimpacto / codigofuente | codigo_impacto / codigo_fuente | |
| fuente39 | fuente_evento | desambiguación de encabezados duplicados del CSV |
| fuente56 | origen_registro | FUENTE39 = fuente del evento; Fuente56 = origen del registro |
| codestror | codigo_estructura_ror | jerarquía '1.1.9.4.1' conservada tal cual |
| codproceso / proceso | codigo_proceso / proceso | homologación solo de grafías |
| codactiviadad / actividad | codigo_actividad / actividad | typo del CSV corregido en el NOMBRE |
| codriesgo / codcausa | cod_riesgo / cod_causa | solo códigos (sin nombre descriptivo en la base) |
| cuenta_p_rdida_8_digitos / documento_p_rdida | cuenta_perdida / documento_perdida | tildes recuperadas |
| cuenta_recuperaci_n_8_digitos / documento_recuperaci_n | cuenta_recuperacion / documento_recuperacion | |
| clase_de_riesgo, idinsumo, subcategorías N2/N3, líneas operativas | mismo nombre | trim | 
| — | id_evento, anio_evento, mes_evento, nivel_severidad, flg_*, _lote_silver | derivados y banderas |

### 1b. Excepciones de formato detectadas al ejecutar la capa
* **Espacio de no-ruptura (\xa0) en codproceso:** 2.243 filas traían el código con un carácter U+00A0 invisible (p. ej. '\xa010.2'), lo que hacía que un mismo proceso figurara con dos códigos. Se normaliza a espacio + trim: el código queda '10.2'. Es la única columna afectada (0 casos en las demás).
* **Fechas de recuperación en MM/dd/yyyy:** 109 de las 235 fechas de recuperación vienen en formato americano (p. ej. '02/20/2026'), conviviendo con dd/MM/yyyy en la misma columna. Se interpretan como MM/dd SOLO cuando el componente central no puede ser mes (>12); el resto se mantiene en dd/MM, formato dominante y verificado de la base. Las otras cuatro columnas de fecha no presentan el problema (0 casos con mes>12 en 2da posición).

### 2. Nulos y campos incompletos
* Marcadores de ausencia de la fuente ('-', vacío) → NULL en: 5 columnas de cuantía, fecha_contabilizacion (~96% ausente), fecha_recuperacion (~99,5%), horas/minutos asociados, cuenta/documento de pérdida y recuperación (~96-99%), idinsumo (23.026 nulos).
* **Nunca se imputan valores.** El ausente queda NULL y se cuenta en las banderas.
* La ausencia de cuantía (25.188 eventos sin bruta, 48,5%) es estructural de la fuente: se documenta como limitación para el modelado (no es severidad 0).

### 3. Homologación de categorías, procesos, causas y severidad
* **Categorías (N1/N2/N3):** 7 / 23 / 51 valores oficiales ya consistentes en la fuente; solo trim. Códigos y nombres verificados 1:1.
* **Proceso:** 36 grafías canónicas; solo se unificaron variantes de escritura (tildes/mayúsculas/punto final/espacios). Tras normalizar el espacio de no-ruptura, los códigos 10.2, 6.3 y 11.3 quedan asociados a más de un nombre genuinamente distinto (9.406 filas) → bandera flg_proceso_ambiguo, sin reetiquetar.
* **Actividad:** se unificaron 4 pares de grafía (p. ej. 'APLICACION MOVIL'→'APLICACIÓN MÓVIL'); 6 códigos quedan con nombres genuinamente distintos (7.727 filas) → flg_actividad_ambigua.
* **Causas:** la base solo trae código (codcausa: 524 códigos; codriesgo: 59). No existe nombre descriptivo y no se inventa; quedan como código tipado en texto. Limitación documentada para el análisis.
* **Severidad:** no existe columna de severidad en la fuente. Nivel DERIVADO de la cuantía bruta con cortes de la propia base, con percentiles EXACTOS: BAJA < p50 (367.371), MEDIA < p90 (14.217.315), ALTA < p99 (152.000.000), EXTREMA >= p99. NULL cuando no hay cuantía. (Versión inicial con percentile_approx: 367.000 / 14.197.184 / 150.124.492; se pasó a exactos el 21-Nov-2026 por consistencia con Gold: mediana exacta 367.371.)
* **Canal:** 13 grafías → 7 canónicos (OTROS, AGENCIA, CAJERO ELECTRÓNICO, INTERNET, POS, APLICACIÓN MÓVIL, IVR); 'CAJERO AUTOMÁTICO' (1 fila) se unifica con 'CAJERO ELECTRÓNICO' como sinónimo real de la base.

### 4. Montos: negativos, ceros, extremos y recuperaciones
* **Negativos:** 28 cuantías netas negativas (todas tipo R / código pérdida A, todas con recuperación registrada, 7 con bruta). Son recuperaciones que superan la pérdida reportada: se conservan intactas y se marcan flg_neta_negativa. No hay negativos en bruta ni en recuperaciones.
* **Ceros:** no existen montos en cero en ninguna columna de cuantía.
* **Extremos:** 268 eventos con cuantía bruta >= p99 exacto (152.000.000 COP) marcados flg_monto_extremo; máximo 7.000.000.000 COP. Se conservan: son los eventos que dominan la cola de severidad.
* **Recuperaciones:** registradas en 235 eventos (total, seguros y/o otras). Se conservan las tres columnas y flg_con_recuperacion.

### 5. Disponibilidad de montos brutos y netos, y tratamiento
* Con bruta: 26.701 (51,5%) | con neta: 26.588 (51,2%) | ambas: 26.567 | bruta sin neta: 134 | neta sin bruta: 21.
* Consistencia neta = bruta − recuperada: 26.566 consistentes, 1 con recuperación no registrada (flg_inconsistencia_neta).
* **Tratamiento acordado:** la severidad analítica se mide sobre la CUANTÍA BRUTA (pérdida registrada); los eventos sin cuantía se excluyen del modelado de severidad (no se imputan como 0). El VaR se calculará en Gold SOLO con codigo_perdida='A' (2.067 eventos), sobre el valor normalizado.

### 6. Trazabilidad
* id_evento: identificador único 1..51.889 por fila (la base no tiene clave natural única; IDINSUMO no es clave: 512 valores repetidos y 23.026 nulos).
* flg_requiere_revision: OR de las excepciones operativas (neta negativa, monto extremo, inconsistencia neta, neta sin bruta, proceso/actividad ambiguo). La ausencia de cuantía NO se considera anomalía del registro.
* Se conservan _archivo_origen, _fecha_ingesta, _lote_procesamiento (Bronze) y se agrega _lote_silver.
* Bronze permanece intacta: todo valor original es recuperable desde allí.

### 7. Reglas invariantes
* 51.889 filas en Silver == 51.889 en Bronze: ninguna eliminada (las filas con valores idénticos son eventos distintos, regla del 20-Nov-2026).
* Sin deduplicación, sin imputación, sin corrección de valores: solo tipado, homologación de grafías y banderas.
* Reconstrucción idempotente: cada ejecución sobrescribe Silver completa desde Bronze.